# ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ — ಅಜೂರ್ ಓಪನ್‌ಎಐ (ಪ್ರತಿಕ್ರಿಯಾ API)

ಈ ಕೋಡ್ ಉದಾಹರಣೆಯಲ್ಲಿ, ನೀವು **ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ (MAF)** ಅನ್ನು ಬಳಸಿಕೊಂಡು **ಅಜೂರ್ ಓಪನ್‌ಎಐ** ಬದ್ಧವಿರುವ ಸರಳ ಏಜೆಂಟ್ ರಚಿಸಲು **ಪ್ರತಿಕ್ರಿಯಾ API** ಬಳಸಿ.

> **ಅಕ್ರಮಾಂತರ ಟಿಪ್ಪಣಿ:** ಈ ಉದಾಹರಣೆಯು ಮುಂಚೆ ಗಿತ್ಹಬ್ ಮಾದರಿಗಳೊಂದಿಗೆ ಸೆಮ್ಯಾಂಟಿಕ್ ಕರ್ಣಲ್ ಅನ್ನು ಬಳಸಿತ್ತು. ಇದು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ಗೆ ಅಕ್ರಮಾಂತರಗೊಳ್ಳಲಾಗಿದೆ ಮತ್ತು ಗಿತ್ಹಬ್ ಮಾದರಿಗಳು (ಪ್ರಾಚೀನಗೊಂಡ, ಜುಲೈ 2026 ರಲ್ಲಿ ನಿವೃತ್ತಿ) ಅಜೂರ್ ಓಪನ್‌ಎಐ ಮೂಲಕ ಬದಲಾಗಿದೆ, ಇದು ಪ್ರತಿಕ್ರಿಯಾ API ಅನ್ನು ಬೆಂಬಲಿಸುತ್ತದೆ. MAF ನಲ್ಲಿ `OpenAIChatClient` ಅಜೂರ್ ಓಪನ್‌ಎಐನ ಸ್ಥಿರ `/openai/v1/` ಅಂತಿಮ ಬಿಂದುವನ್ನು ಗುರಿ ಮಾಡುತ್ತದೆ ಮತ್ತು ಡೀಫಾಲ್ಟಾಗಿ ಪ್ರತಿಕ್ರಿಯಾ API ಅನ್ನು ಬಳಸುತ್ತದೆ.

ಈ ಉದಾಹರಣೆಯ ಉದ್ದೇಶವು ವಿವಿಧ ಏಜಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಜಾರಿಗೊಳಿಸುವಾಗ ನಂತರ ಅನ್ವಯಿಸಲ್ಪಡುವ ಹಂತಗಳನ್ನು ತೋರಿಸುವುದಾಗಿದೆ.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## ಅಗತ್ಯವಿರುವ Python ಪ್ಯಾಕೇಜುಗಳನ್ನು ಆಮದುಮಾಡಿ


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ಒಂದು ಸಾಧನವನ್ನು ನಿರ್ವಚಿಸುವುದು

ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್‌ನಲ್ಲಿ, **ಸಾಧನ** ಎಂದರೆ ಏಜೆಂಟ್ ಕರೆಮಾಡಬಹುದಾದ `@tool` ಅಲಂಕೃತ ಸರಳ Python ಫಂಕ್ಷನ್ ಆಗಿದೆ. ಕೆಳಗೆ ನಾವು ಯಾದೃಚ್ಛಿಕ ರಜಾ ಗಮ್ಯಸ್ಥಾನವನ್ನು ಹಿಂತೆಗೆದು, ಹಿಂದಿನದನ್ನು ಪುನರಾವರ್ತಿಸುವುದನ್ನ ತಪ್ಪಿಸುವ ಸಾಧನವನ್ನು ನಿರ್ವಚಿಸುತ್ತೇವೆ.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ಏಜೆಂಟ್ ರಚನೆ

ಇಲ್ಲಿ, ನಾವು `TravelAgent` ಎಂಬ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ.

ಈ ಉದಾಹರಣೆಯಲ್ಲಿ, ನಾವು ಅತ್ಯಂತ ಸರಳ ಸೂಚನೆಗಳನ್ನು ಬಳಸುತ್ತಿದ್ದೇವೆ. ಏಜೆಂಟ್ ನ ವರ್ತನೆ ಹೇಗೆ ಬದಲಾಗುತ್ತದೆ ಎಂದು ಗಮನಿಸಲು ಈ ಸೂಚನೆಗಳನ್ನು ನೀವು ಬದಲಾಯಿಸಬಹುದು.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ಏಜೆಂಟ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡುವುದು

ಈಗ ನಾವು ಏಜೆಂಟ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡಬಹುದು. ನಮ್ಮ ಬಳಿ ಎಜೆಂಟ್ ಸಂಭಾಷಣೆಯನ್ನು ಸ್ಮರಿಸುವಂತೆ `AgentSession` ರಚಿಸುತ್ತೇವೆ, ನಂತರ ಎರಡು `user_inputs` ಗಳನ್ನು ಕಳುಹಿಸುತ್ತೇವೆ. ಮೊದಲದವು ಒಂದು ಪ್ರವಾಸವನ್ನು ಕೇಳುತ್ತದೆ; ಎರಡನೇದವು ಉಪಯೋಗಿಸುವವರು ಸೂಚನೆಯನ್ನು ಇಷ್ಟಪಡಲಿಲ್ಲ ಮತ್ತು ಇನ್ನೊಂದನ್ನು ಕೇಳುತ್ತಾರೆ — ಏಜೆಂಟ್ ಸೆಷನ್ ಹಿಸ್ಟು ಮತ್ತು `get_random_destination` ಉಪಕರಣವನ್ನು ಬಳಸಿಕೊಂಡು ಪ್ರತಿಕ್ರಿಯಿಸುತ್ತದೆ.

ನೀವು ಈ ಸಂದೇಶಗಳನ್ನು ಪರಿಷ್ಕರಿಸಬಹುದು ಏಜೆಂಟ್ ಹೊಡೆತನ್ನು ವಿಭಿನ್ನವಾಗಿ ಹೊಂದಿಸುತ್ತಾನೆ ಎಂದು ನೋಡುವುದಕ್ಕೆ. ಪ್ರತಿಕ್ರಿಯೆಗಳು **ಸಂಚಿಕೆಯಿಂದ** ಟೋಕೆನ್-ಮಾತ್ರ ಟೋಕೆನ್ ಆಗಿ ಬರುತ್ತವೆ.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
